# G1 Academy Bonus - Task 13: final task - perception-based delivery

## Introduction
The capstone from `notes.txt` section 11: combine perception, grasping, arm control, and SLAM navigation into one pickup -> carry -> drop-off pipeline. Every building block below was developed natively in an earlier task; this notebook assembles them one more time into a single consolidated `AcademyRobot` class - proof that, piece by piece, you could have written `sdk_wrapper.py` yourself - and then defines the end-to-end `deliver(...)` sequence:

1. `extend_arm_forward` (Task 8's `interpolate_to_ll_pose`) to a saved pre-grasp pose
2. open the hand (Task 11's `gradual_open`)
3. perception + pose estimation (Task 12's `DeliveryPipeline`)
4. IK approach in small increments (Task 10's `ik_move_ee`, via `execute_incremental_ik`)
5. close the hand at the grip pose (Task 11's `gradual_close`)
6. interpolate to a `stable_hold_pose` (Task 8's `interpolate_to_ll_pose`)
7. SLAM-navigate pickup -> dropdown while holding that pose (Task 7's `navigate_to_point`)
8. drop-off: IK to the target ArUco pose, open the hand, interpolate back to the extended pose, release ownership (Task 8's `release_arms`)

In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1


## Task 1 - `AcademyRobot`: one consolidated class over everything built in Tasks 2-12
Owns the `ChannelFactory`, the `rt/lowstate` subscriber, `rt/arm_sdk`/Dex3 publishers, `LocoClient`, `G1ArmActionClient`, the native `SlamRpc`, and a `DeliveryPipeline` instance - the same set of native components each earlier task built and exercised independently.

In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1


## Task 2 - `deliver(pickup_point, dropdown_point, ...)`: the end-to-end sequence
Runs the eight-step flow from `notes.txt` section 11. `stable_hold_pose` must already exist in `ll_poses.json` (save it with Task 8's `save_current_ll_pose("stable_hold_pose")` while holding a good carrying posture), and `pickup_point`/`dropdown_point` must already exist in `slam_points.json` (Task 7's `add_point`).

In [ ]:
def deliver(pickup_point, dropdown_point, side="right", marker_length_m=0.04, grasp_threshold=0.5, prompt="the delivery package", require_object_detection=False):
    robot = _robot

    robot.extend_arm_forward(hand=side)
    robot.gradual_open(hand=side)

    frame = robot.get_rgbd()
    if frame is None or time.time() - frame["timestamp"] > 1.0:
        raise RuntimeError("RGB-D frame is unavailable or stale.")
    rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
    if require_object_detection:
        detection = robot.pipeline.detect_object(frame["rgb_jpeg"], prompt)
        if detection.confidence < 0.5:
            raise RuntimeError(f"Low-confidence detection ({detection.confidence}); aborting.")
    pickup_target = robot.pipeline.marker_to_palm_target(robot.pipeline.aruco_pose(rgb, marker_length_m))

    robot.pipeline.execute_incremental_ik(robot.ik_increment, robot.current_palm_xyz(side), pickup_target, side=side)
    robot.gradual_close(hand=side)
    robot.interpolate_to_pose("stable_hold_pose", duration_s=3.0)
    nav = robot.navigate_to_point(dropdown_point)
    if not nav.get("arrived", False):
        raise RuntimeError(f"Navigation to {dropdown_point!r} did not complete: {nav}")

    frame = robot.get_rgbd()
    if frame is None:
        raise RuntimeError("No RGB-D frame received at drop-off.")
    rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
    dropdown_target = robot.pipeline.marker_to_palm_target(robot.pipeline.aruco_pose(rgb, marker_length_m))
    robot.pipeline.execute_incremental_ik(robot.ik_increment, robot.current_palm_xyz(side), dropdown_target, side=side)
    robot.gradual_open(hand=side)
    robot.extend_arm_forward(hand=side)
    robot.release_arms()
    return {"pickup": pickup_point, "dropdown": dropdown_point, "side": side}

_robot = AcademyRobot(iface="eth0", domain_id=0)
# deliver("pickup", "dropdown")


## Reflect
Every method on `AcademyRobot` above is a native rebuild of something `sdk_wrapper.G1` already provides. Compare this class against `sdk_wrapper.py` end to end: same DDS init guard, same publisher/subscriber patterns, same ease-curve interpolation, same IK/SLAM/hand boundaries. Task 1 showed you the finished API; this notebook is the proof you could have built it.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.